# LeetCode 3953 — Maximum Score with Co-Prime Element bằng Python

- Link: https://leetcode.com/problems/maximum-score-with-co-prime-element/
- Độ khó: **Hard**
- Chủ đề: Lý thuyết số (gcd, nguyên tố cùng nhau), hàm Möbius / bao hàm – loại trừ, sàng (sieve), đếm theo bội số

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxScore(self, nums: List[int], maxVal: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Bài cho mảng `nums` và số `maxVal`. Ta được phép đổi bất kỳ phần tử nào thành một số nguyên dương `≤ maxVal` (mỗi lần đổi tốn **1**). Cuối cùng phải chọn **một** chỉ số `i` sao cho `nums[i]` **nguyên tố cùng nhau** (gcd = 1) với **mọi** phần tử khác. Điểm = `giá trị nums[i] − số phần tử đã đổi`. Cần điểm lớn nhất.

Đây là bài "đếm theo ước chung" điển hình: với `n` và giá trị tới `10^5`, ta **không thể** tính gcd của mọi cặp (giá trị, phần tử). Chìa khóa là biến câu hỏi "có bao nhiêu phần tử **không** nguyên tố cùng nhau với `v`?" thành một phép đếm theo **bội số** bằng **bao hàm – loại trừ** (hàm Möbius).

Sau notebook này bạn sẽ:

1. Biến bài toán thành: với mỗi giá trị ứng viên `v`, tính `cnt(v)` = số phần tử của mảng có chung ước nguyên tố với `v`.
2. Hiểu vì sao "đổi phần tử vướng thành `1`" luôn tối ưu và vì sao có hai trường hợp: **giữ nguyên** `nums[i]` hoặc **đổi** `nums[i]` thành `v ≤ maxVal`.
3. Dùng hàm Möbius `μ` để đếm số phần tử nguyên tố cùng nhau với `v` trong `O(V log V)` bằng một cái sàng.
4. Cài đặt 4 cách (vét cạn theo định nghĩa, đếm bằng gcd, Möbius + phân tích thừa số, Möbius + sàng điều hòa) và đối chiếu bằng test ngẫu nhiên có seed.
5. Phân tích độ phức tạp `O(V log V)` với `V = max(maxVal, max(nums))`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums` (độ dài `n`) và `maxVal`.

- Một lần **thay đổi** = đổi một phần tử thành số nguyên dương bất kỳ `≤ maxVal`, tốn 1.
- Sau khi thay đổi, phải chọn một chỉ số `i` sao cho `gcd(nums[i], nums[j]) = 1` với mọi `j ≠ i`.
- `score = nums[i] (giá trị cuối) − tổng số phần tử đã thay đổi`.
- Trả về điểm lớn nhất.

**Ví dụ 1:** `nums = [3,4,6]`, `maxVal = 5` → `4` (đổi `6` thành `5`, chọn `5`: `5 − 1 = 4`).

**Ví dụ 2:** `nums = [1,2,3]`, `maxVal = 4` → `3` (không cần đổi, chọn `3`).

**Ví dụ 3:** `nums = [2,2]`, `maxVal = 1` → `1` (đổi một số `2` thành `1`, chọn số `2` còn lại: `2 − 1 = 1`).

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^5`, `1 ≤ maxVal ≤ 10^5`.

Lưu ý: phần tử **được chọn** có thể giữ nguyên giá trị ban đầu, kể cả khi giá trị đó lớn hơn `maxVal` (không đổi thì không bị giới hạn bởi `maxVal`).

In [1]:
from math import gcd
import random, time

EXAMPLES = [
    (([3, 4, 6], 5), 4),
    (([1, 2, 3], 4), 3),
    (([2, 2], 1), 1),
]
for (nums, mv), expected in EXAMPLES:
    print("nums =", nums, " maxVal =", mv, " -> kỳ vọng", expected)

nums = [3, 4, 6]  maxVal = 5  -> kỳ vọng 4
nums = [1, 2, 3]  maxVal = 4  -> kỳ vọng 3
nums = [2, 2]  maxVal = 1  -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. `gcd` và "nguyên tố cùng nhau"

`gcd(a, b)` là ước chung lớn nhất. Hai số **nguyên tố cùng nhau** (co-prime) khi `gcd = 1`, tức là **không có chung ước nguyên tố nào**. Số `1` nguyên tố cùng nhau với mọi số.

In [2]:
from math import gcd
print(gcd(12, 18))   # 6  -> không co-prime
print(gcd(8, 15))    # 1  -> co-prime
print(gcd(1, 99))    # 1  -> 1 co-prime với mọi số
print(gcd(7, 7))     # 7  -> một số > 1 không co-prime với chính nó

6
1
1
7


### 3.2. "Có chung ước nguyên tố" ⇔ "gcd > 1"

`12 = 2² · 3` và `18 = 2 · 3²` có chung `2` và `3`. `8 = 2³` và `15 = 3 · 5` không chung ước nguyên tố nào nên gcd = 1.
Với bài này, chỉ **tập các ước nguyên tố khác nhau** của `v` (gọi là `rad(v)`) mới quan trọng; số mũ không ảnh hưởng.

In [3]:
def prime_factors(x):
    # trả về tập các ước nguyên tố khác nhau của x
    res, p = set(), 2
    while p * p <= x:
        while x % p == 0:
            res.add(p)
            x //= p
        p += 1
    if x > 1:
        res.add(x)
    return res

for v in (12, 18, 8, 15, 30, 1):
    print(v, "->", sorted(prime_factors(v)))
print("12 và 18 chung:", prime_factors(12) & prime_factors(18))
print("8 và 15 chung :", prime_factors(8) & prime_factors(15))

12 -> [2, 3]
18 -> [2, 3]
8 -> [2]
15 -> [3, 5]
30 -> [2, 3, 5]
1 -> []
12 và 18 chung: {2, 3}
8 và 15 chung : set()


### 3.3. Đếm bội số với mảng tần suất

Gọi `freq[x]` = số lần `x` xuất hiện, và `mult[d]` = số phần tử của mảng **chia hết cho `d`**. Khi đó `mult[d] = freq[d] + freq[2d] + freq[3d] + ...`. Tổng các `V/d` với `d = 1..V` là `V·(1 + 1/2 + 1/3 + ...) ≈ V ln V` (chuỗi điều hòa), nên tính mọi `mult[d]` tốn `O(V log V)`.

Trong Python, `freq[d::d]` là lát cắt lấy các chỉ số `d, 2d, 3d, ...` nên `sum(freq[d::d])` chính là `mult[d]`.

In [4]:
nums = [3, 4, 6, 6, 10]
V = 10
freq = [0] * (V + 1)
for x in nums:
    freq[x] += 1
print("freq =", freq)
mult = [0] * (V + 1)
for d in range(1, V + 1):
    mult[d] = sum(freq[d::d])
print("mult =", mult)
print("số phần tử chia hết cho 2:", mult[2], "(4, 6, 6, 10)")
print("số phần tử chia hết cho 3:", mult[3], "(3, 6, 6)")
print("số phần tử chia hết cho 6:", mult[6], "(6, 6)")

freq = [0, 0, 0, 1, 1, 0, 2, 0, 0, 0, 1]
mult = [0, 5, 4, 3, 1, 1, 2, 0, 0, 0, 1]
số phần tử chia hết cho 2: 4 (4, 6, 6, 10)
số phần tử chia hết cho 3: 3 (3, 6, 6)
số phần tử chia hết cho 6: 2 (6, 6)


### 3.4. Bao hàm – loại trừ và hàm Möbius `μ`

Muốn đếm số phần tử **không** chia hết cho `2` **và không** chia hết cho `3` (tức nguyên tố cùng nhau với `6`), ta có thể làm:

```
coprime(6) = tất cả − (chia hết cho 2) − (chia hết cho 3) + (chia hết cho cả 2 và 3 = chia hết cho 6)
```

Tổng quát, với `v` có các ước nguyên tố khác nhau `p1, p2, ..., pk`:

```
coprime(v) = Σ over d | rad(v) :  μ(d) · mult[d]
```

trong đó `μ(d)` (Möbius) bằng `+1` nếu `d` là tích của số **chẵn** các nguyên tố phân biệt, `−1` nếu là tích của số **lẻ** các nguyên tố phân biệt, và `0` nếu `d` có thừa số bình phương. `μ(1) = 1`.

In [5]:
def mobius_upto(V):
    # sàng tuyến tính: mu[d] cho d = 1..V
    mu = [0] * (V + 1)
    mu[1] = 1
    is_comp = [False] * (V + 1)
    primes = []
    for i in range(2, V + 1):
        if not is_comp[i]:
            primes.append(i)
            mu[i] = -1
        for p in primes:
            if i * p > V:
                break
            is_comp[i * p] = True
            if i % p == 0:
                mu[i * p] = 0
                break
            mu[i * p] = -mu[i]
    return mu

mu = mobius_upto(30)
print([(d, mu[d]) for d in range(1, 16)])
# kiểm tra bằng tay: mu(6)=+1 (2*3), mu(4)=0, mu(30)=-1 (2*3*5)
assert mu[6] == 1 and mu[4] == 0 and mu[30] == -1 and mu[1] == 1

[(1, 1), (2, -1), (3, -1), (4, 0), (5, -1), (6, 1), (7, -1), (8, 0), (9, 0), (10, 1), (11, -1), (12, 0), (13, -1), (14, 1), (15, 1)]


In [6]:
# Kiểm tra công thức bao hàm - loại trừ với nums ở 3.3: đếm phần tử co-prime với 6
nums = [3, 4, 6, 6, 10]
direct = sum(1 for x in nums if gcd(x, 6) == 1)
via_mobius = mu[1] * len(nums) + mu[2] * mult[2] + mu[3] * mult[3] + mu[6] * mult[6]
print("trực tiếp:", direct, " Möbius:", via_mobius)
assert direct == via_mobius

trực tiếp: 0  Möbius: 0


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `nums = [3,4,6]`, `maxVal = 5`.

Thử từng cách chọn:

- Giữ nguyên `3` làm số được chọn: `3` chung ước với `6` → phải đổi `6` thành `1` (cost 1). Điểm `3 − 1 = 2`.
- Giữ nguyên `4`: `4` chung ước `2` với `6` → đổi `6` (cost 1). Điểm `4 − 1 = 3`.
- Giữ nguyên `6`: vướng cả `3` lẫn `4` → cost 2. Điểm `6 − 2 = 4`.
- Đổi một phần tử thành `5` (`5 ≤ maxVal`): `5` nguyên tố cùng nhau với `3`, `4`, `6`. Đổi `6` thành `5` (cost 1), chọn nó: `5 − 1 = 4`.

Đáp án `4`. Hai điều quan sát quan trọng:

1. Phần tử **vướng** (chung ước nguyên tố với số được chọn) chỉ có một cách khắc phục là đổi nó; đổi thành `1` là rẻ nhất (cost 1, và `1` an toàn với mọi giá trị).
2. Số được chọn có thể **giữ nguyên** hoặc **đổi** sang một giá trị tự do `v ≤ maxVal`. Khi đổi, ta nên chọn `i` là một phần tử vốn đã vướng với `v` (đằng nào cũng phải đổi nó) để "một công đôi việc".

In [7]:
nums, maxVal = [3, 4, 6], 5
# giữ nguyên từng phần tử được chọn
for i, x in enumerate(nums):
    cost = sum(1 for j, y in enumerate(nums) if j != i and gcd(x, y) > 1)
    print(f"giữ nguyên nums[{i}]={x}: cost={cost}, điểm={x - cost}")
# đổi thành v
for v in range(1, maxVal + 1):
    cnt = sum(1 for y in nums if gcd(v, y) > 1)       # số phần tử vướng với v
    print(f"đổi thành v={v}: cnt(v)={cnt}")

giữ nguyên nums[0]=3: cost=1, điểm=2
giữ nguyên nums[1]=4: cost=1, điểm=3
giữ nguyên nums[2]=6: cost=2, điểm=4
đổi thành v=1: cnt(v)=0
đổi thành v=2: cnt(v)=2
đổi thành v=3: cnt(v)=2
đổi thành v=4: cnt(v)=2
đổi thành v=5: cnt(v)=0


## 5. Cách 1 — Brute Force theo đúng định nghĩa

**Ý tưởng.** Với mảng cực nhỏ, liệt kê **mọi** cách xử lý từng phần tử: giữ nguyên (cost 0) hoặc đổi thành một giá trị `1..maxVal` (cost 1). Với mỗi tổ hợp, kiểm tra xem có chỉ số `i` nào nguyên tố cùng nhau với mọi phần tử còn lại không, rồi lấy `max` của `giá trị − cost`.

Đây là cách "dịch thẳng đề", không dùng bất kỳ nhận xét thông minh nào, nên dùng làm **chuẩn đối chiếu**.

In [8]:
from itertools import product

def brute_definition(nums, maxVal):
    n = len(nums)
    # mỗi phần tử: (giá trị cuối, cost)
    options = [[(x, 0)] + [(v, 1) for v in range(1, maxVal + 1)] for x in nums]
    best = None
    for choice in product(*options):
        vals = [c[0] for c in choice]
        cost = sum(c[1] for c in choice)
        for i in range(n):
            if all(gcd(vals[i], vals[j]) == 1 for j in range(n) if j != i):
                score = vals[i] - cost
                if best is None or score > best:
                    best = score
    return best

for (args, expected) in EXAMPLES:
    got = brute_definition(*args)
    print(args, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

([3, 4, 6], 5) -> 4 (kỳ vọng 4 )
([1, 2, 3], 4) -> 3 (kỳ vọng 3 )
([2, 2], 1) -> 1 (kỳ vọng 1 )


**Độ phức tạp.** Có `(maxVal + 1)^n` tổ hợp, mỗi tổ hợp kiểm tra `O(n²)` cặp gcd → `O((maxVal+1)^n · n² · log)`: chỉ dùng được cho `n ≤ 4`, `maxVal ≤ 6`.

**Nút thắt.** Số tổ hợp bùng nổ theo `n`. Nhưng ví dụ ở mục 4 cho thấy ta chỉ cần quan tâm **số phần tử vướng**, không cần liệt kê từng cách đổi.

## 6. Bước đột phá

**Quan sát 1 — đổi phần tử vướng thành `1`.** Giả sử số được chọn có giá trị cuối là `v`. Mọi phần tử `y` khác có `gcd(v, y) > 1` bắt buộc phải đổi; đổi thành `1` (hợp lệ vì `1 ≤ maxVal`) luôn tốn đúng 1 và không gây vướng thêm. Phần tử không vướng thì để yên. Vậy với `v` cố định, chi phí cho các phần tử khác = số phần tử vướng.

**Quan sát 2 — hai kiểu chọn số cuối.**

- **(a) Giữ nguyên `nums[i] = x`:** cost = số `j ≠ i` có `gcd(x, nums[j]) > 1`. Gọi `cnt(x)` là số phần tử **trong cả mảng** có `gcd > 1` với `x`. Nếu `x > 1` thì chính `nums[i]` cũng được đếm (vì `gcd(x, x) = x > 1`), nên cost = `cnt(x) − 1`. Nếu `x = 1` thì `cnt(1) = 0` và cost = `0`.
- **(b) Đổi `nums[i]` thành `v` (`1 ≤ v ≤ maxVal`):** cost = `1` (đổi chính nó) `+` số `j ≠ i` vướng với `v`. Nếu `cnt(v) ≥ 1` ta chọn `i` là **một phần tử vướng** (nó vừa phải đổi, vừa là người được chọn) → cost = `1 + (cnt(v) − 1) = cnt(v)`. Nếu `cnt(v) = 0` thì cost = `1`. Gộp lại: **cost = `max(cnt(v), 1)`**.

Đáp án = `max` của `x − (cnt(x) − [x>1])` qua các `x` trong mảng, và `v − max(cnt(v), 1)` qua `v = 1..maxVal`.

**Phần còn thiếu:** tính `cnt(v)` cho **mọi** `v ≤ V = max(maxVal, max(nums))` thật nhanh. `cnt(v) = n − coprime(v)` và `coprime(v) = Σ_{d | rad(v)} μ(d)·mult[d]` đã học ở mục 3.4.

**Quy tắc:** nếu `d` squarefree (μ ≠ 0) thì ta cộng `μ(d)·mult[d]` vào `coprime[v]` cho **mọi bội** `v` của `d`. Đó là một cái sàng điều hòa `O(V log V)`.

In [9]:
# Minh họa quan sát: cost cho hai kiểu chọn với nums = [3,4,6], maxVal = 5
def cnt_naive(nums, v):
    return sum(1 for y in nums if gcd(v, y) > 1)

nums, maxVal = [3, 4, 6], 5
print("(a) giữ nguyên:", [(x, x - (cnt_naive(nums, x) - (1 if x > 1 else 0))) for x in nums])
print("(b) đổi thành v:", [(v, v - max(cnt_naive(nums, v), 1)) for v in range(1, maxVal + 1)])

(a) giữ nguyên: [(3, 2), (4, 3), (6, 4)]
(b) đổi thành v: [(1, 0), (2, 0), (3, 1), (4, 2), (5, 4)]


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Công thức (a)/(b) + đếm `cnt` bằng gcd

**Giải thích từng đoạn.**

1. Với mỗi giá trị khác nhau `x` trong mảng: `cnt(x)` = số phần tử có `gcd(x, y) > 1` (vòng lặp `n` phần tử) → điểm kiểu (a).
2. Với mỗi `v = 1..maxVal`: tương tự → điểm kiểu (b).
3. Lấy `max`.

Cách này đúng nhưng chậm: `O((maxVal + d) · n · log)` với `d` là số giá trị khác nhau.

In [10]:
def semi_brute(nums, maxVal):
    n = len(nums)
    best = -10**18
    for x in set(nums):
        cnt = sum(1 for y in nums if gcd(x, y) > 1)
        best = max(best, x - (cnt - (1 if x > 1 else 0)))
    for v in range(1, maxVal + 1):
        cnt = sum(1 for y in nums if gcd(v, y) > 1)
        best = max(best, v - max(cnt, 1))
    return best

for (args, expected) in EXAMPLES:
    assert semi_brute(*args) == expected
print("Cách 2 khớp 3 ví dụ")

Cách 2 khớp 3 ví dụ


**Độ phức tạp:** `O((maxVal + D) · n · log V)` — với `n = V = 10^5` là cỡ `10^10`, quá chậm. **Không gian:** `O(1)` thêm.

### 7.2. Cách 3 — Möbius + phân tích thừa số bằng SPF

**Ý tưởng.** Dùng mảng `spf[v]` (smallest prime factor) để phân tích `v` thành các ước nguyên tố khác nhau trong `O(log v)`. Với mỗi `v`, liệt kê các **ước squarefree** của `rad(v)` (tối đa `2^ω(v)`, `ω ≤ 6` khi `v ≤ 10^5`), cộng `μ(d)·mult[d]` với dấu theo số nguyên tố đã chọn.

Giải thích từng đoạn:

1. Tính `freq`, `mult` như mục 3.3.
2. Sàng `spf`.
3. Với mỗi `v`: lấy danh sách nguyên tố `ps`; khởi tạo danh sách `(d, sign) = [(1, +1)]`; với mỗi nguyên tố `p`, nhân đôi danh sách bằng cách thêm `(d·p, −sign)`.
4. `coprime(v) = Σ sign · mult[d]`.

In [11]:
def count_coprime_spf(nums, V):
    freq = [0] * (V + 1)
    for x in nums:
        freq[x] += 1
    mult = [0] + [sum(freq[d::d]) for d in range(1, V + 1)]
    spf = list(range(V + 1))
    for i in range(2, int(V ** 0.5) + 1):
        if spf[i] == i:
            for j in range(i * i, V + 1, i):
                if spf[j] == j:
                    spf[j] = i
    coprime = [0] * (V + 1)
    for v in range(1, V + 1):
        ps, x = [], v
        while x > 1:
            p = spf[x]
            ps.append(p)
            while x % p == 0:
                x //= p
        terms = [(1, 1)]
        for p in ps:
            terms += [(d * p, -s) for d, s in terms]
        coprime[v] = sum(s * mult[d] for d, s in terms)
    return coprime

def solve_spf(nums, maxVal):
    n = len(nums)
    V = max(max(nums), maxVal)
    coprime = count_coprime_spf(nums, V)
    best = -10**18
    for x in set(nums):
        cnt = n - coprime[x]
        best = max(best, x - (cnt - (1 if x > 1 else 0)))
    for v in range(1, maxVal + 1):
        cnt = n - coprime[v]
        best = max(best, v - max(cnt, 1))
    return best

for (args, expected) in EXAMPLES:
    assert solve_spf(*args) == expected
print("Cách 3 khớp 3 ví dụ")

Cách 3 khớp 3 ví dụ


**Độ phức tạp:** sàng `O(V log log V)`, `mult` `O(V log V)`, mỗi `v` mất `O(2^ω(v))` → tổng `O(V log V)` (trung bình `2^ω` rất nhỏ). **Không gian:** `O(V)`.

### 7.3. Cách 4 — Möbius + sàng điều hòa (không cần phân tích thừa số)

**Ý tưởng.** Đảo thứ tự cộng: thay vì "với mỗi `v` duyệt các ước squarefree `d`", làm "với mỗi `d` squarefree duyệt các **bội** `v` của `d`":

```
for d in 1..V:
    if mu[d] != 0:
        for v in d, 2d, 3d, ...:
            coprime[v] += mu[d] * mult[d]
```

Vòng lặp trong có `V/d` bước → tổng `O(V log V)` và mã rất ngắn. Đây là lời giải ta dùng để nộp.

In [12]:
def solve_sieve(nums, maxVal):
    n = len(nums)
    V = max(max(nums), maxVal)
    freq = [0] * (V + 1)
    for x in nums:
        freq[x] += 1
    mult = [0] * (V + 1)
    for d in range(1, V + 1):
        mult[d] = sum(freq[d::d])
    mu = mobius_upto(V)
    coprime = [0] * (V + 1)
    for d in range(1, V + 1):
        if mu[d]:
            c = mu[d] * mult[d]
            for v in range(d, V + 1, d):
                coprime[v] += c
    best = -10**18
    for x in set(nums):
        cnt = n - coprime[x]
        best = max(best, x - (cnt - (1 if x > 1 else 0)))
    for v in range(1, maxVal + 1):
        cnt = n - coprime[v]
        best = max(best, v - max(cnt, 1))
    return best

for (args, expected) in EXAMPLES:
    assert solve_sieve(*args) == expected
print("Cách 4 khớp 3 ví dụ")

Cách 4 khớp 3 ví dụ


**Độ phức tạp:** `O(V log V)` thời gian, `O(V)` bộ nhớ, `V = max(maxVal, max(nums)) ≤ 10^5` → khoảng vài triệu thao tác.

Cả ba cách (2, 3, 4) dùng **cùng công thức (a)/(b)**; khác nhau duy nhất ở cách tính `cnt`. Nếu công thức (a)/(b) sai thì cả ba cùng sai — vì thế Cách 1 (vét cạn theo định nghĩa, không dùng công thức) là bằng chứng độc lập.

## 8. TRACE từng bước

Trace Cách 4 trên `nums = [3,4,6]`, `maxVal = 5` (`V = 6`). Bảng các đại lượng:

| `d` | `μ(d)` | `mult[d]` | cộng `μ·mult` vào các bội `v` |
|---|---|---|---|
| 1 | +1 | 3 | mọi `v = 1..6` |
| 2 | −1 | 2 (`4, 6`) | `v = 2, 4, 6` |
| 3 | −1 | 2 (`3, 6`) | `v = 3, 6` |
| 4 | 0 | — | bỏ qua |
| 5 | −1 | 0 | `v = 5` (cộng 0) |
| 6 | +1 | 1 (`6`) | `v = 6` |

Ví dụ `coprime[6] = 3 − 2 − 2 + 1 = 0` (không phần tử nào co-prime với 6), `coprime[5] = 3` (cả ba).

In [13]:
def trace_sieve(nums, maxVal):
    n = len(nums)
    V = max(max(nums), maxVal)
    freq = [0] * (V + 1)
    for x in nums:
        freq[x] += 1
    mult = [0] * (V + 1)
    for d in range(1, V + 1):
        mult[d] = sum(freq[d::d])
    mu = mobius_upto(V)
    print("freq =", freq)
    print("mult =", mult)
    coprime = [0] * (V + 1)
    for d in range(1, V + 1):
        if mu[d] == 0:
            print(f"d={d}: mu=0 -> bỏ qua")
            continue
        c = mu[d] * mult[d]
        for v in range(d, V + 1, d):
            coprime[v] += c
        print(f"d={d}: mu={mu[d]:+d}, mult={mult[d]}, cộng {c:+d} vào bội {list(range(d, V + 1, d))} -> coprime={coprime[1:]}")
    print("--- ứng viên ---")
    best = -10**18
    for x in sorted(set(nums)):
        cnt = n - coprime[x]
        sc = x - (cnt - (1 if x > 1 else 0))
        print(f"(a) giữ x={x}: cnt={cnt}, điểm={sc}")
        best = max(best, sc)
    for v in range(1, maxVal + 1):
        cnt = n - coprime[v]
        sc = v - max(cnt, 1)
        print(f"(b) đổi v={v}: cnt={cnt}, điểm={sc}")
        best = max(best, sc)
    return best

print("đáp án:", trace_sieve([3, 4, 6], 5))

freq = [0, 0, 0, 1, 1, 0, 1]
mult = [0, 3, 2, 2, 1, 0, 1]
d=1: mu=+1, mult=3, cộng +3 vào bội [1, 2, 3, 4, 5, 6] -> coprime=[3, 3, 3, 3, 3, 3]
d=2: mu=-1, mult=2, cộng -2 vào bội [2, 4, 6] -> coprime=[3, 1, 3, 1, 3, 1]
d=3: mu=-1, mult=2, cộng -2 vào bội [3, 6] -> coprime=[3, 1, 1, 1, 3, -1]
d=4: mu=0 -> bỏ qua
d=5: mu=-1, mult=0, cộng +0 vào bội [5] -> coprime=[3, 1, 1, 1, 3, -1]
d=6: mu=+1, mult=1, cộng +1 vào bội [6] -> coprime=[3, 1, 1, 1, 3, 0]
--- ứng viên ---
(a) giữ x=3: cnt=2, điểm=2
(a) giữ x=4: cnt=2, điểm=3
(a) giữ x=6: cnt=3, điểm=4
(b) đổi v=1: cnt=0, điểm=0
(b) đổi v=2: cnt=2, điểm=0
(b) đổi v=3: cnt=2, điểm=1
(b) đổi v=4: cnt=2, điểm=2
(b) đổi v=5: cnt=0, điểm=4
đáp án: 4


## 9. Code nộp LeetCode

Gộp Cách 4 vào `class Solution` đúng chữ ký `maxScore(self, nums, maxVal)`.

In [14]:
from typing import List

class Solution:
    def maxScore(self, nums: List[int], maxVal: int) -> int:
        n = len(nums)
        V = max(max(nums), maxVal)

        # mu[d]: hàm Möbius, sàng tuyến tính
        mu = [0] * (V + 1)
        mu[1] = 1
        comp = [False] * (V + 1)
        primes = []
        for i in range(2, V + 1):
            if not comp[i]:
                primes.append(i)
                mu[i] = -1
            for p in primes:
                ip = i * p
                if ip > V:
                    break
                comp[ip] = True
                if i % p == 0:
                    break
                mu[ip] = -mu[i]

        # mult[d]: số phần tử chia hết cho d
        freq = [0] * (V + 1)
        for x in nums:
            freq[x] += 1
        # coprime[v]: số phần tử nguyên tố cùng nhau với v
        coprime = [0] * (V + 1)
        for d in range(1, V + 1):
            if mu[d]:
                c = mu[d] * sum(freq[d::d])
                if c:
                    for v in range(d, V + 1, d):
                        coprime[v] += c

        best = -10**18
        for x in set(nums):                       # (a) giữ nguyên nums[i] = x
            cnt = n - coprime[x]
            best = max(best, x - (cnt - (1 if x > 1 else 0)))
        for v in range(1, maxVal + 1):            # (b) đổi nums[i] thành v
            cnt = n - coprime[v]
            best = max(best, v - max(cnt, 1))
        return best

sol = Solution()
for (args, expected) in EXAMPLES:
    got = sol.maxScore(*args)
    print(args, "->", got)
    assert got == expected

([3, 4, 6], 5) -> 4
([1, 2, 3], 4) -> 3
([2, 2], 1) -> 1


## 10. Vì sao thuật toán đúng

**Bước 1 — chuẩn hóa phương án tối ưu.** Xét một phương án tối ưu với chỉ số được chọn `i` và giá trị cuối `v`.

- Mọi `j ≠ i` có `gcd(v, nums[j]) > 1` buộc phải bị đổi (không thì vi phạm). Đổi `nums[j]` thành `1` làm cost đúng bằng 1 cho mỗi `j` như vậy — không thể rẻ hơn, và không làm hỏng gì (vì `gcd(v, 1) = 1`). Các `j` còn lại không cần đổi. Cost của phần "các phần tử khác" bằng số `j ≠ i` vướng với `v` (khi `nums[j]` chưa bị đổi). Việc đổi `nums[j]` thành giá trị khác `1` chỉ có thể không tốt hơn.
- Chỉ có hai khả năng cho `nums[i]`: **không đổi** (`v = nums[i]`, cost 0 cho chính nó) hoặc **đổi** (`v ≤ maxVal` tùy ý, cost 1).

**Bước 2 — công thức cost.**

- Kiểu (a): cost = số `j ≠ i` vướng với `x`. Vì `gcd(x, x) = x`, phần tử `i` thuộc tập "vướng với `x`" khi và chỉ khi `x > 1`. Suy ra cost = `cnt(x) − [x > 1]`.
- Kiểu (b): cost = `1 +` số `j ≠ i` vướng với `v`. Số này = `cnt(v) − [nums[i] vướng với v]`. Để cost nhỏ nhất, chọn `i` thuộc tập vướng nếu tập đó không rỗng → cost = `cnt(v)`; nếu rỗng thì cost = `1`.

**Bước 3 — đếm `cnt` đúng.** `coprime(v) = #{y : gcd(y, v) = 1} = Σ_{d | v} μ(d)·#{y : d | y}` là công thức nghịch đảo Möbius (vì `Σ_{d | g} μ(d) = [g = 1]`, áp dụng với `g = gcd(y, v)`). Chỉ các `d` squarefree mới có `μ ≠ 0`, và `d | v` ⇔ `v` là bội của `d`, nên sàng cộng `μ(d)·mult[d]` vào mọi bội `v` của `d` cho kết quả đúng.

**Kết luận.** `max` của hai họ ứng viên (a) và (b) bao trùm mọi phương án khả thi và mỗi ứng viên đạt được thật sự, nên đáp án đúng.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`, `V = max(maxVal, max(nums)) ≤ 10^5`.

| Bước | Thời gian | Ghi chú |
|---|---|---|
| Sàng Möbius tuyến tính | `O(V)` | mỗi hợp số bị đánh dấu một lần |
| Tính `mult[d]` cho mọi `d` | `O(V log V)` | `Σ V/d` |
| Sàng điều hòa cộng vào `coprime` | `O(V log V)` | chỉ `d` squarefree, `Σ_{d sqfree} V/d ≈ 0.6 V ln V` |
| Duyệt ứng viên (a), (b) | `O(n + maxVal)` | |

**Tổng:** `O(n + V log V)` thời gian, `O(V)` không gian. Với `V = 10^5`: khoảng `1.2 · 10^6` bước cho mỗi sàng — rất nhanh.

## 12. Bộ test

Gồm: ví dụ đề, edge case (`n = 1`, toàn số `1`, `maxVal = 1`), phản ví dụ cho cách làm sai, và đối chiếu 4 cách trên input ngẫu nhiên có seed.

In [15]:
# 12.1 Ví dụ đề + edge case
sol = Solution()
for (args, expected) in EXAMPLES:
    assert sol.maxScore(*args) == expected

assert sol.maxScore([7], 1) == 7                 # n=1: giữ nguyên, không ai để vướng
assert sol.maxScore([1, 1, 1], 1) == 1           # toàn số 1: chọn 1, cost 0
assert sol.maxScore([6, 6, 6], 1) == 4           # giữ một số 6, đổi hai số còn lại thành 1: 6-2 = 4
assert brute_definition([6, 6, 6], 1) == 4
print("edge cases OK")

edge cases OK


In [16]:
# 12.2 Brute theo định nghĩa trên input rất nhỏ (seed cố định)
random.seed(3953)
for _ in range(150):
    n = random.randint(1, 4)
    maxVal = random.randint(1, 6)
    nums = [random.randint(1, 12) for _ in range(n)]
    exp = brute_definition(nums, maxVal)
    assert semi_brute(nums, maxVal) == exp, (nums, maxVal)
    assert solve_spf(nums, maxVal) == exp, (nums, maxVal)
    assert solve_sieve(nums, maxVal) == exp, (nums, maxVal)
    assert sol.maxScore(nums, maxVal) == exp, (nums, maxVal)
print("150 test nhỏ: 4 cách khớp vét cạn theo định nghĩa")

150 test nhỏ: 4 cách khớp vét cạn theo định nghĩa


In [17]:
# 12.3 Đối chiếu các cách nhanh với semi_brute trên input lớn hơn
random.seed(2026)
for _ in range(300):
    n = random.randint(1, 25)
    hi = random.choice([10, 30, 60, 100])
    maxVal = random.randint(1, hi)
    nums = [random.randint(1, hi) for _ in range(n)]
    exp = semi_brute(nums, maxVal)
    assert solve_spf(nums, maxVal) == exp, (nums, maxVal)
    assert solve_sieve(nums, maxVal) == exp, (nums, maxVal)
    assert sol.maxScore(nums, maxVal) == exp, (nums, maxVal)
print("300 test vừa: khớp")

# 12.4 Thời gian với dữ liệu lớn
random.seed(1)
big = [random.randint(1, 10**5) for _ in range(10**5)]
t0 = time.time()
r = sol.maxScore(big, 10**5)
print("n = V = 1e5 ->", r, f"({time.time() - t0:.2f}s)")

300 test vừa: khớp


n = V = 1e5 -> 99990 (0.10s)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên trừ chính phần tử được chọn khi đếm `cnt(x)`

**Code sai:** cost kiểu (a) = `cnt(x)` (đếm cả chính nó).
**Vì sao sai:** `gcd(x, x) = x > 1` nên chính `nums[i]` nằm trong `cnt(x)`; nhưng nó đâu cần bị đổi.
**Code đúng:** `cost = cnt(x) − (1 if x > 1 else 0)`.

In [18]:
def wrong_forgot_self(nums, maxVal):
    best = -10**18
    for x in set(nums):
        cnt = sum(1 for y in nums if gcd(x, y) > 1)
        best = max(best, x - cnt)                  # SAI: đếm cả chính nó
    for v in range(1, maxVal + 1):
        cnt = sum(1 for y in nums if gcd(v, y) > 1)
        best = max(best, v - max(cnt, 1))
    return best

print("sai :", wrong_forgot_self([1, 2, 3], 4), " đúng:", Solution().maxScore([1, 2, 3], 4))
print("sai :", wrong_forgot_self([2, 2], 1), " đúng:", Solution().maxScore([2, 2], 1))

sai : 3  đúng: 3
sai : 0  đúng: 1


### Lỗi 2 — Cho rằng số được chọn luôn phải `≤ maxVal`

**Code sai:** chỉ xét `v = 1..maxVal` (bỏ nhóm giữ nguyên).
**Vì sao sai:** phần tử được chọn có thể giữ nguyên giá trị gốc, kể cả lớn hơn `maxVal` (ví dụ `nums = [100, 7]`, `maxVal = 1`).
**Code đúng:** xét cả hai nhóm (a) và (b).

In [19]:
def wrong_only_changed(nums, maxVal):
    best = -10**18
    for v in range(1, maxVal + 1):
        cnt = sum(1 for y in nums if gcd(v, y) > 1)
        best = max(best, v - max(cnt, 1))
    return best

print("sai :", wrong_only_changed([100, 7], 1), " đúng:", Solution().maxScore([100, 7], 1))

sai : 0  đúng: 100


### Lỗi 3 — Đếm cost đổi `nums[i]` là `cnt(v) + 1` khi `cnt(v) ≥ 1`

**Code sai:** `cost = cnt(v) + 1` cho kiểu (b).
**Vì sao sai:** nếu có phần tử vướng với `v`, ta chọn chính phần tử đó làm `i` — nó phải đổi rồi, nên không tốn thêm cho "đổi `i`".
**Code đúng:** `cost = max(cnt(v), 1)`.

In [20]:
def wrong_plus_one(nums, maxVal):
    best = -10**18
    for x in set(nums):
        cnt = sum(1 for y in nums if gcd(x, y) > 1)
        best = max(best, x - (cnt - (1 if x > 1 else 0)))
    for v in range(1, maxVal + 1):
        cnt = sum(1 for y in nums if gcd(v, y) > 1)
        best = max(best, v - (cnt + 1))            # SAI
    return best

print("sai :", wrong_plus_one([3, 4, 6], 5), " đúng:", Solution().maxScore([3, 4, 6], 5))

sai : 4  đúng: 4


### Lỗi 4 — Tính gcd cho mọi cặp (TLE)

`n = 10^5` và `V = 10^5` cho `10^10` lần `gcd`. Đây là lý do cần sàng Möbius (Cách 4). Hãy nhớ: **đếm theo bội số** thay vì **kiểm tra từng cặp**.

### Lỗi 5 — Dùng `mu` không đúng dấu / quên bỏ `μ = 0`

Nếu cộng cả `d` có thừa số bình phương (như `d = 4`) với dấu `±1`, kết quả sai. Kiểm tra nhanh: `coprime[v]` luôn nằm trong `[0, n]`.

## 14. Mẹo & nhận dạng pattern

- Thấy câu hỏi "**bao nhiêu phần tử nguyên tố cùng nhau với `v`**" cho **nhiều `v`** → nghĩ ngay **Möbius + đếm bội số**.
- Ràng buộc `10^5` cho cả giá trị và độ dài là dấu hiệu của **sàng / chuỗi điều hòa** `O(V log V)`.
- Khi một phần tử "vướng" bất kỳ cách sửa nào cũng tốn như nhau, hãy chọn cách sửa **vô hại nhất** (ở đây là đổi thành `1`).
- Luôn tách bài thành các **nhóm trường hợp** (giữ nguyên / đổi) rồi tối ưu từng nhóm.
- Khi nghi ngờ công thức, viết một brute force **không dùng công thức** (như `brute_definition`) và so sánh trên input nhỏ.

## 15. Biến thể

1. **Nhiều số được chọn:** cần chọn `m` chỉ số sao cho chúng đôi một nguyên tố cùng nhau với nhau và với phần còn lại — khó hơn nhiều (bài toán tập độc lập trên đồ thị ước chung).
2. **Chi phí đổi không đều:** đổi phần tử `j` tốn `w[j]` → thay `cnt(v)` bằng **tổng trọng số** các phần tử vướng; vẫn dùng Möbius nhưng `mult[d]` là tổng trọng số các phần tử chia hết cho `d`.
3. **Không được đổi thành `1`:** chỉ đổi thành số nguyên tố lớn hợp lệ → cần tập các giá trị đích, bài chuyển sang xét xem `maxVal` có đủ số nguyên tố không.
4. **Đếm số cặp co-prime** trong mảng: `Σ_d μ(d) · C(mult[d], 2)` — cùng kỹ thuật Möbius.

In [21]:
# Biến thể 2: chi phí đổi theo trọng số w[j] (chỉ để minh họa kỹ thuật; không phải đề gốc)
def weighted_noncoprime_sum(nums, w, V):
    wm = [0] * (V + 1)                   # wm[x] = tổng trọng số các phần tử có giá trị x
    for x, c in zip(nums, w):
        wm[x] += c
    wmult = [0] + [sum(wm[d::d]) for d in range(1, V + 1)]
    mu = mobius_upto(V)
    cop = [0] * (V + 1)
    for d in range(1, V + 1):
        if mu[d]:
            for v in range(d, V + 1, d):
                cop[v] += mu[d] * wmult[d]
    total = sum(w)
    return [total - cop[v] for v in range(V + 1)]

nums, w = [3, 4, 6], [5, 1, 2]
res = weighted_noncoprime_sum(nums, w, 6)
for v in range(1, 7):
    assert res[v] == sum(c for x, c in zip(nums, w) if gcd(v, x) > 1)
print("trọng số vướng theo v:", res[1:])

trọng số vướng theo v: [0, 3, 7, 3, 0, 8]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn theo định nghĩa | thử mọi cách giữ/đổi từng phần tử | `O((maxVal+1)^n · n²)` | `O(n)` | chỉ để kiểm chứng, `n ≤ 4` |
| 2. Công thức + gcd | cost = số phần tử vướng, đếm bằng gcd | `O((maxVal + D)·n·log V)` | `O(1)` | `n, V ≲ 2000`; dễ cài |
| 3. Möbius + SPF | `coprime(v)` qua ước squarefree của `v` | `O(V log V)` | `O(V)` | khi đã có sẵn SPF |
| 4. Möbius + sàng điều hòa | cộng `μ(d)·mult[d]` vào mọi bội | `O(V log V)` | `O(V)` | **lời giải nộp** — ngắn, nhanh |

`D` = số giá trị khác nhau trong mảng, `V = max(maxVal, max(nums))`.

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `semi_brute([2, 3, 5], 1)` bằng bao nhiêu?

**Bài 2 (tính tay).** `nums = [4, 6, 9]`, `maxVal = 7`. Điểm lớn nhất là bao nhiêu? (Liệt kê kiểu (a) và (b).)

**Bài 3 (điền TODO).** Hoàn thiện `my_coprime_count(nums, v)` — đếm số phần tử nguyên tố cùng nhau với `v` chỉ bằng `math.gcd`.

**Bài 4 (tự viết lại).** Viết `my_max_score(nums, maxVal)` dùng công thức (a)/(b) và `cnt` bằng gcd (Cách 2), không nhìn code.

**Bài 5 (biến thể).** Viết `count_coprime_pairs(nums)` đếm số cặp `i < j` có `gcd(nums[i], nums[j]) = 1` bằng Möbius: `Σ_d μ(d)·C(mult[d], 2)`.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_coprime_count(nums, v):
    # TODO: trả về số phần tử y trong nums có gcd(v, y) == 1
    pass

# Bài 4 — TODO
def my_max_score(nums, maxVal):
    # TODO 1: best = rất nhỏ
    # TODO 2: với mỗi x khác nhau: cnt = số y có gcd>1; best = max(best, x - (cnt - (x>1)))
    # TODO 3: với mỗi v trong 1..maxVal: best = max(best, v - max(cnt, 1))
    pass

# Bài 5 — TODO
def count_coprime_pairs(nums):
    # TODO: freq, mult, mu rồi Σ mu[d] * mult[d]*(mult[d]-1)//2
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `maxVal = 1` nên chỉ đổi được thành `1`.
- Mức 2: `[2,3,5]` đôi một nguyên tố cùng nhau, chọn `5` không cần đổi.
- Mức 3: kiểu (a) với `x = 5`: `cnt(5) = 1` (chính nó) → cost 0 → điểm `5`.

**Bài 2.**
- Mức 1: tính `cnt(x)` cho `x = 4, 6, 9` (trong mảng `[4,6,9]`).
- Mức 2: `4` vướng `4,6`; `6` vướng cả ba; `9` vướng `6,9`.
- Mức 3: (a) `4−1 = 3`, `6−2 = 4`, `9−1 = 8`; (b) `v = 5` → `cnt = 0`, điểm `4`; `v = 7` → điểm `6`. Max = `8`.

**Bài 3.**
- Mức 1: dùng `sum(1 for y in nums if ...)`.
- Mức 2: điều kiện `gcd(v, y) == 1`.
- Mức 3: `return sum(1 for y in nums if gcd(v, y) == 1)`.

**Bài 4.**
- Mức 1: xem lại hàm `semi_brute`.
- Mức 2: hai vòng lặp ứng viên (a) rồi (b).
- Mức 3: nhớ `(1 if x > 1 else 0)` và `max(cnt, 1)`.

**Bài 5.**
- Mức 1: số cặp cùng chia hết cho `d` là `C(mult[d], 2)`.
- Mức 2: cặp có `gcd = 1` ⇔ `Σ_{d | gcd} μ(d) = 1`.
- Mức 3: `Σ_d μ[d] * mult[d]*(mult[d]-1)//2`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
assert semi_brute([2, 3, 5], 1) == 5
# Bài 2
assert semi_brute([4, 6, 9], 7) == 8 == Solution().maxScore([4, 6, 9], 7)

# Bài 3
def my_coprime_count_ans(nums, v):
    return sum(1 for y in nums if gcd(v, y) == 1)

assert my_coprime_count_ans([3, 4, 6], 6) == 0
assert my_coprime_count_ans([3, 4, 6], 5) == 3
assert my_coprime_count_ans([1, 2, 3], 4) == 2

# Bài 4
def my_max_score_ans(nums, maxVal):
    best = -10**18
    for x in set(nums):
        cnt = sum(1 for y in nums if gcd(x, y) > 1)
        best = max(best, x - (cnt - (1 if x > 1 else 0)))
    for v in range(1, maxVal + 1):
        cnt = sum(1 for y in nums if gcd(v, y) > 1)
        best = max(best, v - max(cnt, 1))
    return best

random.seed(7)
for _ in range(60):
    n = random.randint(1, 8)
    mv = random.randint(1, 15)
    arr = [random.randint(1, 30) for _ in range(n)]
    assert my_max_score_ans(arr, mv) == Solution().maxScore(arr, mv)
print("Bài 1–4 OK")

Bài 1–4 OK


In [24]:
# Bài 5
def count_coprime_pairs_ans(nums):
    V = max(nums)
    freq = [0] * (V + 1)
    for x in nums:
        freq[x] += 1
    mu = mobius_upto(V)
    total = 0
    for d in range(1, V + 1):
        m = sum(freq[d::d])
        total += mu[d] * m * (m - 1) // 2
    return total

random.seed(11)
for _ in range(80):
    arr = [random.randint(1, 40) for _ in range(random.randint(1, 12))]
    exp = sum(1 for i in range(len(arr)) for j in range(i + 1, len(arr)) if gcd(arr[i], arr[j]) == 1)
    assert count_coprime_pairs_ans(arr) == exp, arr
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm theo bội số + Möbius):**

```python
V = max(max(nums), maxVal)
freq[x] += 1 cho mỗi x
mu = sàng tuyến tính
coprime = [0] * (V + 1)
for d in range(1, V + 1):
    if mu[d]:
        c = mu[d] * sum(freq[d::d])      # mult[d]
        for v in range(d, V + 1, d):
            coprime[v] += c
best = max(
    max(x - ((n - coprime[x]) - (x > 1)) for x in set(nums)),       # giữ nguyên
    max(v - max(n - coprime[v], 1) for v in 1..maxVal),             # đổi thành v
)
```

**Checklist trước khi submit:**

- [ ] `V = max(maxVal, max(nums))` (mảng `coprime` phải phủ cả giá trị trong `nums` lẫn `maxVal`).
- [ ] Kiểu (a): trừ chính phần tử khi `x > 1`; với `x = 1` thì không trừ.
- [ ] Kiểu (b): `cost = max(cnt(v), 1)`.
- [ ] Phần tử giữ nguyên có thể `> maxVal`.
- [ ] Bỏ qua `d` có `μ(d) = 0`.
- [ ] Đã đối chiếu vét cạn theo định nghĩa trên input nhỏ (`n ≤ 4`).
- [ ] Chạy thử `n = V = 10^5` trong vài giây.